In [1]:
# ============================================================
# 10_signal_performance_analysis_v0_1
#
# Goal:
#   Analyze priced trade-candidate results from notebook 09.
#
# Scope:
#   Diagnostics only.
#   No new trading rules.
#   No optimization.
#   No risk sizing.
#   No portfolio simulation.
#
# Main question:
#   Which signal dimensions explain performance differences,
#   especially weak front-tenor put performance?
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PRICING_INPUT_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_pricing_v0_1.parquet"

SIGNAL_PERFORMANCE_SUMMARY_CSV = AUDIT_DIR / "signal_performance_summary_v0_1.csv"
SIGNAL_PERFORMANCE_TENOR_TIER_CSV = AUDIT_DIR / "signal_performance_tenor_tier_v0_1.csv"
SIGNAL_PERFORMANCE_BUCKETS_CSV = AUDIT_DIR / "signal_performance_buckets_v0_1.csv"
SIGNAL_PERFORMANCE_TAILS_CSV = AUDIT_DIR / "signal_performance_tail_losses_v0_1.csv"

print("Project root:", PROJECT_ROOT)
print("Pricing input exists:", PRICING_INPUT_PARQUET.exists())

Project root: C:\Users\patri\vrp_project
Pricing input exists: True


In [2]:
# ============================================================
# Load priced trade-candidate results from notebook 09
# ============================================================

pricing_df = pd.read_parquet(PRICING_INPUT_PARQUET).copy()

required_cols = [
    "trade_date",
    "trade_side",
    "selected_tenor",
    "actual_dte",
    "signal_tier",
    "signal_state",
    "pricing_status",
    "entry_credit_points",
    "short_option_pnl_points",
    "short_option_win",
    "short_option_expired_otm",
    "normalized_pnl_dollars",
    "normalized_pnl_pct_notional",
    "signal_primary_vrp",
    "signal_blended_z",
    "signal_3m_z",
    "signal_1y_z",
    "signal_implied_vol",
    "signal_trailing_rv",
    "spx_rsi_14",
    "spx_close",
    "expiry_spx_close",
    "short_strike_selected",
]

missing_cols = [c for c in required_cols if c not in pricing_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

pricing_df["trade_date"] = pricing_df["trade_date"].astype(int)
pricing_df["trade_year"] = pricing_df["trade_date"].astype(str).str[:4].astype(int)

priced_df = pricing_df[pricing_df["pricing_status"] == "priced"].copy()
not_priced_df = pricing_df[pricing_df["pricing_status"] != "priced"].copy()

put_df = priced_df[priced_df["trade_side"] == "put"].copy()
call_df = priced_df[priced_df["trade_side"] == "call"].copy()

print("Total rows:", len(pricing_df))
print("Priced rows:", len(priced_df))
print("Not priced rows:", len(not_priced_df))

print("\nPriced rows by side:")
display(priced_df["trade_side"].value_counts().rename("count").to_frame())

print("\nDate range:")
print(priced_df["trade_date"].min(), "to", priced_df["trade_date"].max())

display(priced_df.tail(20))

Total rows: 1829
Priced rows: 1806
Not priced rows: 23

Priced rows by side:


,count
trade_side,
put,1074
call,732



Date range:
20180823 to 20260604


,trade_date,underlyer,trade_side,option_structure,option_right,selected_tenor,target_expiry_date,expiry_selection_rule,exit_rule,short_strike_rule,...,short_option_pnl_dollars_per_contract,short_option_win,normalized_notional,normalized_contracts,normalized_pnl_dollars,normalized_pnl_pct_notional,normalized_pnl_pct_notional_check,normalization_check_diff,pricing_status,trade_year
1804,20260518,SPX,put,naked_short_put,P,30.0,20260617,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,11530.0,1.0,1000000.0,1.350795,15574.661795,0.015575,0.015575,0.000000e+00,priced,2026
1805,20260518,SPX,call,naked_short_call,C,30.0,20260617,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,40.0,1.0,1000000.0,1.350795,54.031784,0.000054,0.000054,0.000000e+00,priced,2026
1806,20260519,SPX,put,naked_short_put,P,27.0,20260615,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,11560.0,1.0,1000000.0,1.359876,15720.170093,0.015720,0.015720,-3.469447e-18,priced,2026
1807,20260519,SPX,call,naked_short_call,C,27.0,20260615,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,45.0,1.0,1000000.0,1.359876,61.194434,0.000061,0.000061,-1.355253e-20,priced,2026
1810,20260526,SPX,put,naked_short_put,P,18.0,20260613,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,-504.0,0.0,1000000.0,1.329943,-670.291204,-0.000670,-0.000670,0.000000e+00,priced,2026
1811,20260526,SPX,call,naked_short_call,C,18.0,20260613,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,65.0,1.0,1000000.0,1.329943,86.446286,0.000086,0.000086,-1.355253e-20,priced,2026
1812,20260527,SPX,put,naked_short_put,P,12.0,20260608,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,-8486.0,0.0,1000000.0,1.329724,-11284.034275,-0.011284,-0.011284,0.000000e+00,priced,2026
1813,20260527,SPX,call,naked_short_call,C,12.0,20260608,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,30.0,1.0,1000000.0,1.329724,39.891707,0.000040,0.000040,0.000000e+00,priced,2026
1814,20260528,SPX,put,naked_short_put,P,12.0,20260609,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,-6344.0,0.0,1000000.0,1.322116,-8387.507057,-0.008388,-0.008388,0.000000e+00,priced,2026
1815,20260528,SPX,call,naked_short_call,C,12.0,20260609,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,60.0,1.0,1000000.0,1.322116,79.326990,0.000079,0.000079,1.355253e-20,priced,2026


In [3]:
# ============================================================
# Create analysis buckets
# ============================================================

def make_tenor_group(tenor):
    if pd.isna(tenor):
        return np.nan

    tenor = float(tenor)

    if tenor <= 18:
        return "front_9_18d"

    if tenor <= 27:
        return "middle_21_27d"

    return "back_30_33d"


priced_df["tenor_group"] = priced_df["selected_tenor"].apply(make_tenor_group)

# Buckets chosen to line up with existing put thresholds and useful diagnostics.
priced_df["blended_z_bucket"] = pd.cut(
    priced_df["signal_blended_z"],
    bins=[-np.inf, 0.1, 0.65, 0.7, 1.0, 1.5, np.inf],
    labels=[
        "<=0.10",
        "0.10_to_0.65",
        "0.65_to_0.70",
        "0.70_to_1.00",
        "1.00_to_1.50",
        ">1.50",
    ],
)

priced_df["primary_vrp_bucket"] = pd.cut(
    priced_df["signal_primary_vrp"],
    bins=[-np.inf, 0.7, 0.85, 1.0, 1.25, 1.5, np.inf],
    labels=[
        "<=0.70",
        "0.70_to_0.85",
        "0.85_to_1.00",
        "1.00_to_1.25",
        "1.25_to_1.50",
        ">1.50",
    ],
)

priced_df["trailing_rv_bucket"] = pd.cut(
    priced_df["signal_trailing_rv"],
    bins=[-np.inf, 6.8, 8.5, 12.0, 16.0, 25.0, np.inf],
    labels=[
        "<=6.8",
        "6.8_to_8.5",
        "8.5_to_12",
        "12_to_16",
        "16_to_25",
        ">25",
    ],
)

priced_df["rsi_bucket"] = pd.cut(
    priced_df["spx_rsi_14"],
    bins=[-np.inf, 40, 50, 58, 65, 70, 72, 77, np.inf],
    labels=[
        "<=40",
        "40_to_50",
        "50_to_58",
        "58_to_65",
        "65_to_70",
        "70_to_72",
        "72_to_77",
        ">77",
    ],
)

# Refresh side-specific frames after adding buckets.
put_df = priced_df[priced_df["trade_side"] == "put"].copy()
call_df = priced_df[priced_df["trade_side"] == "call"].copy()

print("Tenor group counts:")
display(
    priced_df
    .groupby(["trade_side", "tenor_group"])
    .size()
    .unstack(fill_value=0)
)

print("\nPut tier counts:")
display(put_df["signal_tier"].value_counts().rename("count").to_frame())

Tenor group counts:


tenor_group,back_30_33d,front_9_18d,middle_21_27d
trade_side,,,
call,182,333,217
put,406,380,288



Put tier counts:


,count
signal_tier,
A_strongest,563
C_acceptable,272
B_good,239


In [4]:
# ============================================================
# Reusable performance summary helper
# ============================================================

def summarize_performance(df, group_cols):
    """
    Summarize entry-to-expiry short-option performance.

    Normalized P&L is same-notional P&L from notebook 09.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    summary = (
        df
        .groupby(group_cols, dropna=False, observed=False)
        .agg(
            rows=("trade_date", "count"),
            win_rate=("short_option_win", "mean"),
            expired_otm_rate=("short_option_expired_otm", "mean"),
            avg_entry_credit_points=("entry_credit_points", "mean"),
            median_entry_credit_points=("entry_credit_points", "median"),
            avg_pnl_points=("short_option_pnl_points", "mean"),
            median_pnl_points=("short_option_pnl_points", "median"),
            worst_pnl_points=("short_option_pnl_points", "min"),
            best_pnl_points=("short_option_pnl_points", "max"),
            avg_normalized_pnl_pct=("normalized_pnl_pct_notional", "mean"),
            median_normalized_pnl_pct=("normalized_pnl_pct_notional", "median"),
            worst_normalized_pnl_pct=("normalized_pnl_pct_notional", "min"),
            best_normalized_pnl_pct=("normalized_pnl_pct_notional", "max"),
            avg_normalized_pnl_dollars=("normalized_pnl_dollars", "mean"),
            median_normalized_pnl_dollars=("normalized_pnl_dollars", "median"),
            worst_normalized_pnl_dollars=("normalized_pnl_dollars", "min"),
            best_normalized_pnl_dollars=("normalized_pnl_dollars", "max"),
            avg_signal_primary_vrp=("signal_primary_vrp", "mean"),
            avg_signal_blended_z=("signal_blended_z", "mean"),
            avg_signal_trailing_rv=("signal_trailing_rv", "mean"),
            avg_signal_implied_vol=("signal_implied_vol", "mean"),
            avg_rsi=("spx_rsi_14", "mean"),
            avg_actual_dte=("actual_dte", "mean"),
        )
        .reset_index()
    )

    return summary

In [5]:
# ============================================================
# High-level performance summaries
# ============================================================

summary_by_side_df = summarize_performance(
    priced_df,
    ["trade_side"],
)

summary_by_side_year_df = summarize_performance(
    priced_df,
    ["trade_side", "trade_year"],
)

summary_by_side_tenor_df = summarize_performance(
    priced_df,
    ["trade_side", "selected_tenor"],
)

summary_by_side_tenor_group_df = summarize_performance(
    priced_df,
    ["trade_side", "tenor_group"],
)

print("Summary by side:")
display(summary_by_side_df)

print("\nSummary by side and tenor:")
display(summary_by_side_tenor_df)

print("\nSummary by side and tenor group:")
display(summary_by_side_tenor_group_df)

print("\nSummary by side and year:")
display(summary_by_side_year_df)

Summary by side:


,trade_side,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,call,732,1.000000,1.000000,0.281421,0.25,0.281421,0.25,0.05,1.05,...,63.360051,56.552231,6.596254,281.784510,1.233719,1.290198,8.927872,15.945767,65.695310,20.696721
1,put,1074,0.776536,0.666667,62.054004,57.80,21.109255,45.75,-657.02,214.20,...,4704.203880,10689.293638,-139149.196036,42988.137121,1.151122,1.115686,10.475938,18.542560,56.800615,22.850093



Summary by side and tenor:


,trade_side,selected_tenor,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,call,9.0,69,1.000000,1.000000,0.210870,0.150,0.210870,0.150,0.05,...,47.383644,41.818200,6.596254,137.330883,1.502965,1.586218,7.775714,15.316425,64.414069,9.246377
1,call,12.0,111,1.000000,1.000000,0.239640,0.200,0.239640,0.200,0.05,...,55.899782,49.351521,7.006776,220.027216,1.483030,1.480746,7.701186,15.485104,65.376384,10.693694
2,call,15.0,78,1.000000,1.000000,0.258333,0.250,0.258333,0.250,0.05,...,59.150537,47.355195,8.365443,281.784510,1.289902,1.416928,8.384369,15.434860,65.633761,15.371795
3,call,18.0,75,1.000000,1.000000,0.308000,0.300,0.308000,0.300,0.05,...,68.790532,66.145816,11.241765,262.963318,1.141643,1.128167,9.462054,16.111558,65.639565,17.026667
4,call,21.0,73,1.000000,1.000000,0.303425,0.300,0.303425,0.300,0.05,...,71.653878,66.132609,11.774153,208.508624,1.141621,1.285407,9.484441,16.178742,65.313397,21.821918
5,call,24.0,61,1.000000,1.000000,0.299180,0.300,0.299180,0.300,0.05,...,69.225301,63.823541,16.550975,174.137584,1.113850,1.259478,9.862035,16.434729,65.177252,23.737705
6,call,27.0,83,1.000000,1.000000,0.316867,0.300,0.316867,0.300,0.05,...,67.984834,66.629946,15.268574,200.052514,1.172035,1.264678,9.199455,16.287643,66.299209,27.891566
7,call,30.0,152,1.000000,1.000000,0.295066,0.250,0.295066,0.250,0.05,...,64.652085,60.837094,11.020620,195.484858,1.118254,1.101859,9.332009,16.174928,66.283184,29.888158
8,call,33.0,30,1.000000,1.000000,0.335000,0.225,0.335000,0.225,0.05,...,73.628123,57.678365,11.120526,220.201033,0.999638,1.078885,10.141429,16.343514,67.454938,31.666667
9,put,9.0,87,0.747126,0.643678,42.022989,40.200,5.307356,28.800,-377.16,...,1205.018841,7225.758433,-91511.926686,21945.639952,1.055408,0.976532,11.321213,19.012209,55.766523,9.356322



Summary by side and tenor group:


,trade_side,tenor_group,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,call,back_30_33d,182,1.000000,1.000000,0.301648,0.25,0.301648,0.250,0.05,...,66.131652,60.387344,11.020620,220.201033,1.098702,1.098072,9.465430,16.202717,66.476330,30.181319
1,call,front_9_18d,333,1.000000,1.000000,0.253453,0.25,0.253453,0.250,0.05,...,57.799937,49.272249,6.596254,281.784510,1.365034,1.408243,8.273245,15.579477,65.296547,12.915916
2,call,middle_21_27d,217,1.000000,1.000000,0.307373,0.30,0.307373,0.300,0.05,...,69.567824,66.084572,11.774153,208.508624,1.145447,1.270190,9.481581,16.292355,65.652187,24.682028
3,put,back_30_33d,406,0.798030,0.684729,72.770690,68.50,30.848399,57.095,-657.02,...,6359.204753,13035.518983,-112316.486943,34740.486122,1.174212,1.112002,10.197916,18.313185,56.275388,30.724138
4,put,front_9_18d,380,0.731579,0.657895,49.070000,46.20,6.245737,35.250,-448.88,...,1650.347333,8310.272494,-139149.196036,33689.694072,1.107252,1.034861,11.013003,19.031438,56.397019,13.226316
5,put,middle_21_27d,288,0.805556,0.652778,64.078125,61.30,26.991354,49.800,-441.66,...,6400.506425,11419.468028,-71883.010669,42988.137121,1.176458,1.227525,10.159243,18.220867,58.073563,24.447917



Summary by side and year:


,trade_side,trade_year,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,call,2018,24,1.000000,1.000000,0.279167,0.250,0.279167,0.250,0.15,...,96.077191,86.904749,51.273111,172.157931,1.540579,1.527013,5.598399,11.842546,63.490526,19.958333
1,call,2019,129,1.000000,1.000000,0.144574,0.100,0.144574,0.100,0.05,...,48.976938,36.608447,15.893652,220.027216,1.255540,1.382925,7.434506,13.343782,66.249718,21.162791
2,call,2020,93,1.000000,1.000000,0.257527,0.250,0.257527,0.250,0.05,...,76.139814,68.241498,14.718176,281.784510,1.301670,1.455550,11.955596,22.074934,65.837719,20.548387
3,call,2021,109,1.000000,1.000000,0.150459,0.150,0.150459,0.150,0.05,...,35.509697,32.848058,10.984955,108.653923,1.593978,1.221493,7.907710,17.056281,65.089207,21.412844
4,call,2022,23,1.000000,1.000000,0.386957,0.350,0.386957,0.350,0.10,...,93.921861,84.900557,20.861409,208.508624,0.529240,0.609483,15.720392,20.155049,64.263661,19.608696
5,call,2023,103,1.000000,1.000000,0.385437,0.350,0.385437,0.350,0.05,...,88.011496,78.553009,12.167493,220.201033,0.940820,1.464349,9.741548,14.955837,65.978932,21.145631
6,call,2024,122,1.000000,1.000000,0.406557,0.400,0.406557,0.400,0.10,...,75.593146,73.503354,16.670167,156.796771,1.143682,1.304244,7.976280,13.986003,65.834666,19.934426
7,call,2025,102,1.000000,1.000000,0.314706,0.275,0.314706,0.275,0.05,...,48.901978,43.612575,8.365443,113.088740,1.257418,1.124861,8.727875,15.847774,65.226185,20.892157
8,call,2026,27,1.000000,1.000000,0.370370,0.350,0.370370,0.350,0.05,...,50.680278,50.397275,6.596254,107.465494,1.203038,0.821148,8.877051,15.846159,68.242781,18.666667
9,put,2018,29,0.413793,0.379310,29.903448,28.100,-31.528276,-13.130,-195.31,...,-10536.711523,-4497.545703,-67685.067422,30751.563098,1.155022,0.917577,9.505812,16.390169,54.037987,23.310345


In [6]:
# ============================================================
# Put tier and tenor x tier diagnostics
# ============================================================

put_tier_summary_df = summarize_performance(
    put_df,
    ["signal_tier"],
).sort_values("signal_tier")

put_tenor_tier_summary_df = summarize_performance(
    put_df,
    ["selected_tenor", "signal_tier"],
).sort_values(["selected_tenor", "signal_tier"])

put_tenor_group_tier_summary_df = summarize_performance(
    put_df,
    ["tenor_group", "signal_tier"],
).sort_values(["tenor_group", "signal_tier"])

print("Put tier summary:")
display(put_tier_summary_df)

print("\nPut tenor x tier summary:")
display(put_tenor_tier_summary_df)

print("\nPut tenor group x tier summary:")
display(put_tenor_group_tier_summary_df)

Put tier summary:


,signal_tier,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,A_strongest,563,0.836590,0.710480,68.492540,63.1,33.833268,55.7,-657.02,214.2,...,8316.313704,13095.096845,-112316.486943,42988.137121,1.185862,1.381100,11.374151,20.567882,53.443334,23.040853
1,B_good,239,0.682008,0.573222,48.405439,44.4,5.645230,31.3,-195.31,129.9,...,440.774728,7668.311185,-67685.067422,34740.486122,1.282796,1.174332,7.944368,15.042712,62.645496,23.577406
2,C_acceptable,272,0.735294,0.658088,60.719853,54.7,8.360294,40.1,-651.52,149.1,...,973.842615,9209.368587,-139149.196036,31848.877315,0.963518,0.514787,10.841199,17.425674,58.613934,21.816176



Put tenor x tier summary:


,selected_tenor,signal_tier,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,9.0,A_strongest,43,0.744186,0.604651,49.732558,48.30,3.511860,32.800,-377.16,...,997.414511,8947.463594,-91511.926686,21945.639952,1.104014,1.308935,12.550050,21.663222,54.220630,9.395349
1,9.0,B_good,17,0.764706,0.705882,32.764706,32.20,0.671176,26.900,-134.18,...,-201.347211,6121.394913,-31357.409195,11951.392811,1.174253,0.970153,8.635917,15.275604,56.165751,9.529412
2,9.0,C_acceptable,27,0.740741,0.666667,35.574074,32.60,11.085926,28.500,-230.36,...,2421.137697,6491.644497,-49403.261954,14026.392261,0.903172,0.451168,11.054918,17.142903,57.977134,9.185185
3,12.0,A_strongest,52,0.750000,0.634615,53.121154,53.35,13.628269,39.600,-325.53,...,3762.623719,10216.239578,-97529.773051,26920.188725,1.154168,1.354531,13.434078,23.706069,52.431531,10.884615
4,12.0,B_good,20,0.850000,0.850000,36.630000,34.00,22.815500,32.950,-117.54,...,5631.119713,7542.038821,-26310.370145,18764.539478,1.224176,0.996507,9.115354,16.463174,59.220586,10.800000
5,12.0,C_acceptable,42,0.714286,0.595238,43.119048,37.50,-1.556429,30.380,-197.51,...,338.419260,7375.656871,-43229.534863,22014.218206,0.963694,0.492248,10.354995,16.630299,57.283007,11.880952
6,15.0,A_strongest,51,0.843137,0.764706,61.398039,58.30,27.957059,45.600,-445.88,...,8188.648063,11925.441486,-108475.796818,33689.694072,1.223674,1.451612,12.470789,22.819128,52.818660,15.470588
7,15.0,B_good,17,0.647059,0.588235,42.088235,38.20,8.845294,32.700,-185.25,...,1107.594747,7047.282745,-41120.063928,10486.376345,1.336249,1.204178,7.453353,14.697328,61.975216,16.117647
8,15.0,C_acceptable,23,0.739130,0.739130,54.813043,55.30,-4.959130,44.000,-381.43,...,-2265.757512,8637.213921,-112644.153390,21714.673806,0.972263,0.422523,10.541886,16.898176,58.209829,15.695652
9,18.0,A_strongest,40,0.650000,0.650000,57.542500,55.20,-0.716500,46.750,-272.62,...,1522.621177,10207.175227,-43277.189419,23468.695934,1.166352,1.421658,10.729049,19.237360,55.528818,17.150000



Put tenor group x tier summary:


,tenor_group,signal_tier,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,back_30_33d,A_strongest,212,0.872642,0.745283,79.942453,75.15,48.957736,66.80,-657.02,...,10972.845411,14603.684743,-112316.486943,29954.859691,1.196124,1.300243,10.831380,19.829270,51.944191,30.693396
1,back_30_33d,B_good,94,0.638298,0.521277,57.085106,55.70,7.975106,34.21,-163.86,...,755.918676,10022.457995,-54263.250092,34740.486122,1.346800,1.289928,7.875240,15.434883,64.019434,30.936170
2,back_30_33d,C_acceptable,100,0.790000,0.710000,72.311000,71.80,13.957500,54.70,-651.52,...,1845.375472,11697.330715,-111511.425505,28692.854399,0.965525,0.545679,11.038287,17.804689,58.178121,30.590000
3,front_9_18d,A_strongest,186,0.752688,0.666667,55.558065,53.15,12.133495,43.85,-445.88,...,3855.221267,10526.359076,-108475.796818,33689.694072,1.164252,1.385045,12.383852,22.029591,53.617371,13.145161
4,front_9_18d,B_good,77,0.753247,0.701299,38.877922,36.30,12.460130,30.10,-185.25,...,1779.438007,7344.599424,-41120.063928,18764.539478,1.227122,1.027563,8.356872,15.255292,59.703357,13.545455
5,front_9_18d,C_acceptable,117,0.683761,0.615385,45.463248,42.40,-7.204103,31.96,-448.88,...,-1939.793980,7233.419751,-139149.196036,22014.218206,0.937748,0.482961,10.581758,16.750298,58.639981,13.145299
6,middle_21_27d,A_strongest,165,0.884848,0.715152,68.361818,63.10,38.862182,59.30,-441.66,...,9931.940804,13525.580418,-71883.010669,42988.137121,1.197037,1.480543,10.933322,19.869143,55.173318,24.363636
7,middle_21_27d,B_good,68,0.661765,0.500000,47.195588,45.70,-5.292353,30.95,-195.31,...,-1510.704736,8482.673423,-67685.067422,15185.179609,1.257364,1.180733,7.572826,14.259876,64.077767,24.764706
8,middle_21_27d,C_acceptable,55,0.745455,0.654545,72.100000,72.70,31.293273,53.40,-192.96,...,5587.337085,10145.026354,-43730.014300,31848.877315,1.014688,0.526324,11.034757,18.173264,59.350915,24.309091


In [7]:
# ============================================================
# Signal bucket diagnostics
# ============================================================

put_blended_z_bucket_summary_df = summarize_performance(
    put_df,
    ["blended_z_bucket"],
)

put_primary_vrp_bucket_summary_df = summarize_performance(
    put_df,
    ["primary_vrp_bucket"],
)

put_trailing_rv_bucket_summary_df = summarize_performance(
    put_df,
    ["trailing_rv_bucket"],
)

put_rsi_bucket_summary_df = summarize_performance(
    put_df,
    ["rsi_bucket"],
)

print("Put performance by blended z bucket:")
display(put_blended_z_bucket_summary_df)

print("\nPut performance by primary VRP bucket:")
display(put_primary_vrp_bucket_summary_df)

print("\nPut performance by trailing RV bucket:")
display(put_trailing_rv_bucket_summary_df)

print("\nPut performance by RSI bucket:")
display(put_rsi_bucket_summary_df)

Put performance by blended z bucket:


,blended_z_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=0.10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.10_to_0.65,230,0.765217,0.704348,61.920435,54.95,11.613261,42.30,-651.52,149.1,...,1871.742372,9441.860501,-139149.196036,31848.877315,0.917819,0.379254,11.341192,17.884703,55.774076,21.213043
2,0.65_to_0.70,36,0.805556,0.750000,54.522222,46.85,24.885833,34.00,-137.68,129.9,...,6702.936797,9489.226103,-24040.193719,34740.486122,1.025491,0.677364,10.543919,17.567874,58.578491,21.277778
3,0.70_to_1.00,242,0.743802,0.640496,62.053719,54.25,12.520000,43.35,-657.02,200.1,...,2643.287128,9910.898724,-112316.486943,39527.877920,1.102909,0.844451,10.450201,18.108219,56.630084,23.144628
4,1.00_to_1.50,297,0.734007,0.622896,59.824916,57.50,15.845286,43.90,-445.88,214.2,...,3684.270663,10467.055819,-108475.796818,42988.137121,1.180319,1.246246,10.088958,18.070211,57.739880,23.158249
5,>1.50,269,0.858736,0.695167,65.637546,63.10,42.262119,56.90,-343.15,161.0,...,9838.678901,13340.802988,-52544.486465,28845.059994,1.378553,1.903870,10.177445,20.147741,56.556777,23.855019



Put performance by primary VRP bucket:


,primary_vrp_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=0.70,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.70_to_0.85,185,0.800000,0.756757,58.716757,53.20,13.501892,44.60,-651.52,200.1,...,4096.224181,10145.026354,-111511.425505,39527.877920,0.778551,0.754001,12.650204,18.668841,55.005950,21.529730
2,0.85_to_1.00,220,0.718182,0.600000,59.447727,55.70,5.283773,41.45,-657.02,214.2,...,1245.586298,9504.918453,-139149.196036,42988.137121,0.930096,0.886796,11.071773,17.619460,55.669104,22.131818
3,1.00_to_1.25,310,0.751613,0.638710,57.650323,53.45,16.536258,38.12,-417.26,156.2,...,4081.212365,9745.178210,-80491.232358,34740.486122,1.123563,1.098721,10.094418,17.707437,58.218444,22.938710
4,1.25_to_1.50,209,0.808612,0.693780,66.965550,61.30,35.562919,55.20,-325.53,178.8,...,6631.672248,11996.327348,-97529.773051,33689.694072,1.370622,1.291095,9.653773,19.158724,57.000578,23.110048
5,>1.50,150,0.840000,0.673333,72.250000,70.05,43.014467,60.35,-343.15,161.0,...,9128.594499,13355.014922,-67685.067422,28384.414940,1.685918,1.688129,8.854475,20.608091,57.464789,24.986667



Put performance by trailing RV bucket:


,trailing_rv_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=6.8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6.8_to_8.5,292,0.667808,0.565068,44.395548,39.95,1.764897,28.65,-221.94,98.8,...,-459.903921,7587.222768,-67685.067422,17654.090431,1.263355,1.097765,7.438439,14.078672,65.253115,23.373288
2,8.5_to_12,529,0.818526,0.695652,62.695274,59.80,27.459263,50.01,-441.66,161.0,...,5904.332834,11375.853514,-112644.153390,25103.697999,1.146900,1.160366,9.780562,17.505673,57.480923,23.504726
3,12_to_16,175,0.845714,0.760000,78.424000,70.40,31.330057,62.70,-657.02,178.8,...,8274.115806,16312.211636,-112316.486943,29954.859691,1.083339,1.095502,13.641369,23.660771,46.705176,21.428571
4,16_to_25,75,0.733333,0.626667,84.029333,83.60,22.093067,66.80,-448.88,149.6,...,6744.111638,20299.893636,-139149.196036,34740.486122,0.913742,0.927557,18.946229,30.041182,43.601669,19.626667
5,>25,3,1.000000,1.000000,163.433333,200.10,163.433333,200.10,76.00,214.2,...,36478.734588,39527.877920,26920.188725,42988.137121,0.860275,0.862007,32.336293,49.837432,33.003918,20.000000



Put performance by RSI bucket:


,rsi_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=40,117,0.803419,0.692308,92.971795,83.60,29.841538,69.30,-657.02,214.2,...,7330.395089,16723.057003,-139149.196036,42988.137121,1.064466,0.969812,14.907773,25.316825,33.902183,24.675214
1,40_to_50,170,0.729412,0.629412,76.504706,74.25,15.776235,58.80,-343.15,150.2,...,4328.440611,14260.953184,-70285.451793,33471.893277,1.155138,1.156081,12.416863,21.970774,45.161466,22.323529
2,50_to_58,204,0.789216,0.681373,59.933824,58.90,17.111618,44.70,-445.88,112.2,...,3522.590065,10878.919526,-108475.796818,27379.007313,1.161696,1.182298,10.563128,18.718581,54.700405,21.245098
3,58_to_65,280,0.810714,0.717857,52.643929,52.80,25.418643,44.75,-441.66,115.6,...,5706.357913,10156.996841,-71883.010669,23165.404686,1.189978,1.207989,9.362757,16.967447,61.435478,21.714286
4,65_to_70,198,0.792929,0.651515,52.574242,50.30,26.468636,37.45,-381.43,115.3,...,5469.485140,9775.420280,-112644.153390,21155.308209,1.152083,1.039478,8.855053,15.830573,67.428921,24.388889
5,70_to_72,43,0.674419,0.627907,49.097674,46.00,8.191395,33.90,-154.79,85.3,...,1342.886454,9428.288625,-41048.522378,18145.308889,1.071716,0.980499,8.324741,14.206951,70.838503,25.139535
6,72_to_77,62,0.677419,0.516129,52.819355,49.45,4.788871,32.85,-221.94,89.3,...,27.952796,8339.976530,-52544.486465,14179.226710,1.145382,0.981312,8.199463,14.560987,74.226736,24.758065
7,>77,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ============================================================
# Front-tenor failure diagnostics
# ============================================================

front_put_df = put_df[put_df["tenor_group"] == "front_9_18d"].copy()
nonfront_put_df = put_df[put_df["tenor_group"] != "front_9_18d"].copy()

front_vs_nonfront_summary_df = summarize_performance(
    put_df,
    ["tenor_group"],
)

front_put_tier_summary_df = summarize_performance(
    front_put_df,
    ["signal_tier"],
).sort_values("signal_tier")

front_put_z_bucket_summary_df = summarize_performance(
    front_put_df,
    ["blended_z_bucket"],
)

front_put_rv_bucket_summary_df = summarize_performance(
    front_put_df,
    ["trailing_rv_bucket"],
)

front_put_rsi_bucket_summary_df = summarize_performance(
    front_put_df,
    ["rsi_bucket"],
)

print("Front vs non-front put summary:")
display(front_vs_nonfront_summary_df)

print("\nFront put performance by tier:")
display(front_put_tier_summary_df)

print("\nFront put performance by blended z bucket:")
display(front_put_z_bucket_summary_df)

print("\nFront put performance by trailing RV bucket:")
display(front_put_rv_bucket_summary_df)

print("\nFront put performance by RSI bucket:")
display(front_put_rsi_bucket_summary_df)

Front vs non-front put summary:


,tenor_group,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,back_30_33d,406,0.798030,0.684729,72.770690,68.5,30.848399,57.095,-657.02,178.8,...,6359.204753,13035.518983,-112316.486943,34740.486122,1.174212,1.112002,10.197916,18.313185,56.275388,30.724138
1,front_9_18d,380,0.731579,0.657895,49.070000,46.2,6.245737,35.250,-448.88,113.2,...,1650.347333,8310.272494,-139149.196036,33689.694072,1.107252,1.034861,11.013003,19.031438,56.397019,13.226316
2,middle_21_27d,288,0.805556,0.652778,64.078125,61.3,26.991354,49.800,-441.66,214.2,...,6400.506425,11419.468028,-71883.010669,42988.137121,1.176458,1.227525,10.159243,18.220867,58.073563,24.447917



Front put performance by tier:


,signal_tier,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,A_strongest,186,0.752688,0.666667,55.558065,53.15,12.133495,43.85,-445.88,113.2,...,3855.221267,10526.359076,-108475.796818,33689.694072,1.164252,1.385045,12.383852,22.029591,53.617371,13.145161
1,B_good,77,0.753247,0.701299,38.877922,36.30,12.460130,30.10,-185.25,71.6,...,1779.438007,7344.599424,-41120.063928,18764.539478,1.227122,1.027563,8.356872,15.255292,59.703357,13.545455
2,C_acceptable,117,0.683761,0.615385,45.463248,42.40,-7.204103,31.96,-448.88,94.0,...,-1939.793980,7233.419751,-139149.196036,22014.218206,0.937748,0.482961,10.581758,16.750298,58.639981,13.145299



Front put performance by blended z bucket:


,blended_z_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=0.10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.10_to_0.65,105,0.695238,0.666667,46.002857,43.60,-6.420095,33.9,-448.88,94.0,...,-1743.862694,7809.318280,-139149.196036,22014.218206,0.907488,0.404364,10.834167,16.932016,56.907321,13.104762
2,0.65_to_0.70,15,0.866667,0.866667,42.473333,41.90,26.417333,38.5,-134.18,71.1,...,6584.498986,7898.626129,-23994.220501,18764.539478,0.946085,0.679393,11.757523,18.761846,54.020377,12.466667
3,0.70_to_1.00,83,0.734940,0.662651,46.601205,45.80,7.053735,32.3,-325.53,103.5,...,1089.198499,8156.970268,-97529.773051,27763.427801,1.119677,0.848494,10.651738,18.319112,57.838961,13.120482
4,1.00_to_1.50,101,0.653465,0.584158,50.614851,46.10,-3.552277,32.6,-445.88,113.2,...,-393.005283,7421.184486,-108475.796818,33689.694072,1.160280,1.228532,11.095063,19.537451,56.115437,13.534653
5,>1.50,76,0.855263,0.697368,55.252632,53.05,31.901974,44.6,-272.62,97.6,...,8694.212406,11649.106367,-39615.671891,27103.707811,1.331010,1.922253,11.398621,22.090634,54.960527,13.250000



Front put performance by trailing RV bucket:


,trailing_rv_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=6.8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6.8_to_8.5,100,0.730000,0.640000,36.572000,33.30,10.610600,26.75,-185.25,71.6,...,1335.317965,7087.185996,-43229.534863,10486.376345,1.215167,0.935572,7.489836,13.826849,63.223944,13.740000
2,8.5_to_12,165,0.739394,0.684848,48.915152,45.20,5.043636,36.90,-381.43,97.6,...,1318.996977,8732.681211,-112644.153390,15255.803477,1.106297,0.984184,9.770298,17.094575,57.713541,13.327273
3,12_to_16,72,0.763889,0.666667,55.795833,54.65,13.035833,44.25,-259.40,103.5,...,4370.301167,11828.056208,-60499.029000,27103.707811,1.081876,1.323818,13.644512,23.656755,51.246701,12.500000
4,16_to_25,42,0.642857,0.571429,67.264286,66.60,-12.725238,49.75,-448.88,113.2,...,-1562.290094,17478.711884,-139149.196036,33689.694072,0.904982,0.982390,19.343318,30.527383,43.841215,12.904762
5,>25,1,1.000000,1.000000,76.000000,76.00,76.000000,76.00,76.00,76.0,...,26920.188725,26920.188725,26920.188725,26920.188725,0.795663,0.724210,29.034256,43.220187,54.645172,11.000000



Front put performance by RSI bucket:


,rsi_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=40,29,0.793103,0.655172,62.341379,58.1,22.773448,45.100,-448.88,113.2,...,5721.152085,12263.075235,-139149.196036,33689.694072,1.016574,0.942640,16.636628,27.587568,32.771911,12.448276
1,40_to_50,65,0.538462,0.507692,65.169231,64.7,-28.466154,42.690,-272.62,94.0,...,-4124.949082,10564.790280,-60499.029000,27763.427801,1.113006,1.164865,13.358908,23.132233,45.629746,13.323077
2,50_to_58,93,0.741935,0.688172,49.534409,47.9,-0.314946,38.100,-445.88,80.7,...,-218.314345,8099.703987,-108475.796818,26920.188725,1.116984,1.113828,11.064320,19.127338,54.653613,12.677419
3,58_to_65,115,0.843478,0.791304,40.298261,39.8,25.592609,36.300,-151.51,69.6,...,5705.580683,8456.307095,-43229.534863,20547.174060,1.099133,1.002809,9.643083,16.679034,61.127705,13.365217
4,65_to_70,49,0.775510,0.653061,42.716327,41.9,16.782857,33.300,-381.43,79.6,...,3197.627118,7821.802402,-112644.153390,15251.494024,1.139960,0.910154,9.136760,16.272140,67.345177,13.632653
5,70_to_72,12,0.416667,0.416667,46.716667,42.1,-7.896667,-4.405,-131.65,81.8,...,-3953.359843,-751.923741,-41048.522378,11268.428288,1.063648,0.874674,8.970733,15.146890,70.707361,14.583333
6,72_to_77,17,0.647059,0.352941,41.647059,36.2,-4.600588,12.830,-162.35,44.5,...,-925.981682,2936.592043,-50225.993769,10744.665696,1.178118,0.952447,8.286098,14.840170,73.745551,14.117647
7,>77,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# ============================================================
# Tail-loss diagnostics
# ============================================================

put_df["loss_bucket"] = pd.cut(
    put_df["normalized_pnl_pct_notional"],
    bins=[-np.inf, -0.10, -0.05, -0.02, 0.0, 0.01, 0.02, np.inf],
    labels=[
        "<=-10%",
        "-10%_to_-5%",
        "-5%_to_-2%",
        "-2%_to_0%",
        "0%_to_1%",
        "1%_to_2%",
        ">2%",
    ],
)

put_loss_bucket_summary_df = summarize_performance(
    put_df,
    ["loss_bucket"],
)

put_tail_loss_df = (
    put_df
    .sort_values("normalized_pnl_pct_notional")
    [
        [
            "trade_date",
            "trade_year",
            "selected_expiry_date",
            "selected_tenor",
            "tenor_group",
            "actual_dte",
            "signal_tier",
            "signal_state",
            "spx_close",
            "expiry_spx_close",
            "short_strike_selected",
            "entry_credit_points",
            "expiry_intrinsic_value",
            "short_option_pnl_points",
            "normalized_pnl_dollars",
            "normalized_pnl_pct_notional",
            "signal_primary_vrp",
            "signal_blended_z",
            "signal_3m_z",
            "signal_1y_z",
            "signal_trailing_rv",
            "signal_implied_vol",
            "spx_rsi_14",
        ]
    ]
    .head(50)
)

print("Put loss bucket summary:")
display(put_loss_bucket_summary_df)

print("\nWorst 50 put losses:")
display(put_tail_loss_df)

Put loss bucket summary:


,loss_bucket,rows,win_rate,expired_otm_rate,avg_entry_credit_points,median_entry_credit_points,avg_pnl_points,median_pnl_points,worst_pnl_points,best_pnl_points,...,avg_normalized_pnl_dollars,median_normalized_pnl_dollars,worst_normalized_pnl_dollars,best_normalized_pnl_dollars,avg_signal_primary_vrp,avg_signal_blended_z,avg_signal_trailing_rv,avg_signal_implied_vol,avg_rsi,avg_actual_dte
0,<=-10%,7,0.0,0.000000,91.171429,113.90,-535.210000,-579.42,-657.02,-381.43,...,-112184.664304,-111511.425505,-139149.196036,-100415.930124,0.893704,0.622933,14.435930,22.520001,43.365510,24.714286
1,-10%_to_-5%,18,0.0,0.000000,63.783333,58.70,-290.616111,-260.16,-531.32,-154.64,...,-69682.513319,-68605.237307,-97529.773051,-50225.993769,1.087878,0.859082,11.146413,18.805215,53.191814,25.055556
2,-5%_to_-2%,86,0.0,0.000000,52.918605,50.35,-146.281744,-134.16,-297.16,-60.65,...,-32845.567269,-30734.993952,-49503.281712,-20139.736713,1.146314,1.068837,9.735699,17.041932,55.970382,21.883721
3,-2%_to_0%,129,0.0,0.000000,56.451163,54.10,-41.088527,-35.15,-137.68,-0.82,...,-8751.741764,-8443.658227,-19734.993779,-108.178563,1.122405,1.071902,9.740358,16.922004,59.133054,21.767442
4,0%_to_1%,250,1.0,0.688000,45.875200,42.60,34.080720,33.44,1.48,69.90,...,7234.800732,8046.418653,212.094244,9995.189869,1.087319,0.953119,8.926175,15.390447,61.671455,18.128000
5,1%_to_2%,479,1.0,0.920668,64.078706,62.20,63.014384,61.30,26.00,133.30,...,13969.695480,13525.580418,10017.259467,19962.062391,1.182188,1.225190,10.320080,18.542106,57.003749,24.922756
6,>2%,105,1.0,0.980952,103.466667,93.60,103.099810,93.60,53.20,214.20,...,24241.923916,22861.068030,20038.662122,42988.137121,1.228538,1.172211,16.007923,28.959526,43.605458,26.257143



Worst 50 put losses:


,trade_date,trade_year,selected_expiry_date,selected_tenor,tenor_group,actual_dte,signal_tier,signal_state,spx_close,expiry_spx_close,...,short_option_pnl_points,normalized_pnl_dollars,normalized_pnl_pct_notional,signal_primary_vrp,signal_blended_z,signal_3m_z,signal_1y_z,signal_trailing_rv,signal_implied_vol,spx_rsi_14
390,20200224,2020,20200313,18.0,front_9_18d,18.0,C_acceptable,put_only,3225.89,2711.02,...,-448.88,-139149.196036,-0.139149,0.944174,0.231537,0.086865,0.376210,17.159166,27.511918,36.699503
386,20200219,2020,20200306,15.0,front_9_18d,16.0,C_acceptable,both_put_and_call,3386.15,2972.37,...,-381.43,-112644.153390,-0.112644,0.962067,0.306802,0.188951,0.424653,8.541072,13.817279,65.614116
1486,20250303,2025,20250404,30.0,back_30_33d,32.0,A_strongest,put_only,5849.72,5074.08,...,-657.02,-112316.486943,-0.112316,0.995026,0.957791,1.043115,0.872467,13.868474,22.808456,37.160997
1488,20250305,2025,20250404,30.0,back_30_33d,30.0,C_acceptable,put_only,5842.63,5074.08,...,-651.52,-111511.425505,-0.111511,0.773114,0.437754,0.449960,0.425549,14.787108,21.765203,39.448300
925,20220912,2022,20220930,15.0,front_9_18d,18.0,A_strongest,put_only,4110.41,3585.62,...,-445.88,-108475.796818,-0.108476,0.833395,1.124479,1.505591,0.743367,16.255368,24.658477,54.578509
1487,20250304,2025,20250404,30.0,back_30_33d,31.0,A_strongest,put_only,5778.15,5074.08,...,-582.32,-100779.661310,-0.100780,0.991487,0.930650,1.002726,0.858573,14.517995,23.834459,33.223807
1490,20250307,2025,20250404,30.0,back_30_33d,28.0,C_acceptable,put_only,5770.20,5074.08,...,-579.42,-100415.930124,-0.100416,0.756667,0.371519,0.359000,0.384038,15.922328,23.244212,36.833340
389,20200221,2020,20200306,12.0,front_9_18d,14.0,A_strongest,put_only,3337.75,2972.37,...,-325.53,-97529.773051,-0.097530,1.302942,0.791398,0.781646,0.801150,8.904773,17.082566,53.661884
1489,20250306,2025,20250404,30.0,back_30_33d,29.0,A_strongest,put_only,5738.52,5074.08,...,-531.32,-92588.332880,-0.092588,0.916220,0.737497,0.774625,0.700369,15.864565,25.083189,33.946232
892,20220606,2022,20220617,9.0,front_9_18d,11.0,A_strongest,put_only,4121.43,3674.84,...,-377.16,-91511.926686,-0.091512,0.815093,1.301274,2.276757,0.325791,16.981583,25.525448,50.962658


In [10]:
# ============================================================
# Expectancy decomposition
# ============================================================

def expectancy_decomposition(df, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    working = df.copy()

    working["is_win"] = working["short_option_pnl_points"] > 0
    working["is_loss"] = working["short_option_pnl_points"] <= 0

    grouped = working.groupby(group_cols, dropna=False, observed=False)

    rows = []

    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        wins = group[group["is_win"]].copy()
        losses = group[group["is_loss"]].copy()

        win_rate = len(wins) / len(group) if len(group) > 0 else np.nan
        loss_rate = len(losses) / len(group) if len(group) > 0 else np.nan

        avg_win_pct = wins["normalized_pnl_pct_notional"].mean()
        avg_loss_pct = losses["normalized_pnl_pct_notional"].mean()

        expectancy_pct = group["normalized_pnl_pct_notional"].mean()

        row = {
            col: value
            for col, value in zip(group_cols, group_key)
        }

        row.update({
            "rows": len(group),
            "win_rate": win_rate,
            "loss_rate": loss_rate,
            "avg_win_pct": avg_win_pct,
            "avg_loss_pct": avg_loss_pct,
            "expectancy_pct": expectancy_pct,
            "win_contribution_pct": win_rate * avg_win_pct if pd.notna(avg_win_pct) else np.nan,
            "loss_contribution_pct": loss_rate * avg_loss_pct if pd.notna(avg_loss_pct) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


put_expectancy_by_tenor_df = expectancy_decomposition(
    put_df,
    ["selected_tenor"],
)

put_expectancy_by_tier_df = expectancy_decomposition(
    put_df,
    ["signal_tier"],
)

put_expectancy_by_tenor_group_tier_df = expectancy_decomposition(
    put_df,
    ["tenor_group", "signal_tier"],
)

print("Put expectancy by tenor:")
display(put_expectancy_by_tenor_df)

print("\nPut expectancy by tier:")
display(put_expectancy_by_tier_df)

print("\nPut expectancy by tenor group and tier:")
display(put_expectancy_by_tenor_group_tier_df)

Put expectancy by tenor:


,selected_tenor,rows,win_rate,loss_rate,avg_win_pct,avg_loss_pct,expectancy_pct,win_contribution_pct,loss_contribution_pct
0,9.0,87,0.747126,0.252874,0.008977,-0.021758,0.001205,0.006707,-0.005502
1,12.0,114,0.754386,0.245614,0.011035,-0.022374,0.002829,0.008324,-0.005495
2,15.0,91,0.780220,0.219780,0.012800,-0.026223,0.004223,0.009987,-0.005763
3,18.0,88,0.636364,0.363636,0.010649,-0.024402,-0.002097,0.006777,-0.008874
4,21.0,88,0.806818,0.193182,0.011933,-0.027857,0.004246,0.009628,-0.005381
5,24.0,91,0.802198,0.197802,0.013674,-0.021620,0.006693,0.010970,-0.004276
6,27.0,109,0.807339,0.192661,0.014315,-0.019006,0.007895,0.011557,-0.003662
7,30.0,261,0.819923,0.180077,0.015750,-0.033605,0.006862,0.012914,-0.006052
8,33.0,145,0.758621,0.241379,0.013930,-0.021185,0.005454,0.010567,-0.005114



Put expectancy by tier:


,signal_tier,rows,win_rate,loss_rate,avg_win_pct,avg_loss_pct,expectancy_pct,win_contribution_pct,loss_contribution_pct
0,A_strongest,563,0.836590,0.163410,0.014855,-0.025158,0.008316,0.012427,-0.004111
1,B_good,239,0.682008,0.317992,0.009984,-0.020027,0.000441,0.006809,-0.006369
2,C_acceptable,272,0.735294,0.264706,0.012108,-0.029953,0.000974,0.008903,-0.007929



Put expectancy by tenor group and tier:


,tenor_group,signal_tier,rows,win_rate,loss_rate,avg_win_pct,avg_loss_pct,expectancy_pct,win_contribution_pct,loss_contribution_pct
0,back_30_33d,A_strongest,212,0.872642,0.127358,0.016349,-0.025861,0.010973,0.014267,-0.003294
1,back_30_33d,B_good,94,0.638298,0.361702,0.012186,-0.019414,0.000756,0.007778,-0.007022
2,back_30_33d,C_acceptable,100,0.790000,0.210000,0.014521,-0.045838,0.001845,0.011471,-0.009626
3,front_9_18d,A_strongest,186,0.752688,0.247312,0.013114,-0.024325,0.003855,0.009871,-0.006016
4,front_9_18d,B_good,77,0.753247,0.246753,0.008187,-0.017780,0.001779,0.006167,-0.004387
5,front_9_18d,C_acceptable,117,0.683761,0.316239,0.009085,-0.025777,-0.001940,0.006212,-0.008152
6,middle_21_27d,A_strongest,165,0.884848,0.115152,0.014631,-0.026176,0.009932,0.012946,-0.003014
7,middle_21_27d,B_good,68,0.661765,0.338235,0.009365,-0.022790,-0.001511,0.006198,-0.007708
8,middle_21_27d,C_acceptable,55,0.745455,0.254545,0.013357,-0.017166,0.005587,0.009957,-0.004369


In [11]:
# ============================================================
# Save diagnostic outputs
# ============================================================

summary_by_side_df.to_csv(AUDIT_DIR / "signal_perf_by_side_v0_1.csv", index=False)
summary_by_side_year_df.to_csv(AUDIT_DIR / "signal_perf_by_side_year_v0_1.csv", index=False)
summary_by_side_tenor_df.to_csv(AUDIT_DIR / "signal_perf_by_side_tenor_v0_1.csv", index=False)
summary_by_side_tenor_group_df.to_csv(AUDIT_DIR / "signal_perf_by_side_tenor_group_v0_1.csv", index=False)

put_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tier_v0_1.csv", index=False)
put_tenor_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tenor_tier_v0_1.csv", index=False)
put_tenor_group_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_tenor_group_tier_v0_1.csv", index=False)

put_blended_z_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_blended_z_bucket_v0_1.csv", index=False)
put_primary_vrp_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_primary_vrp_bucket_v0_1.csv", index=False)
put_trailing_rv_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_trailing_rv_bucket_v0_1.csv", index=False)
put_rsi_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_rsi_bucket_v0_1.csv", index=False)

front_vs_nonfront_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_front_vs_nonfront_v0_1.csv", index=False)
front_put_tier_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_tier_v0_1.csv", index=False)
front_put_z_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_z_bucket_v0_1.csv", index=False)
front_put_rv_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_rv_bucket_v0_1.csv", index=False)
front_put_rsi_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_front_put_rsi_bucket_v0_1.csv", index=False)

put_loss_bucket_summary_df.to_csv(AUDIT_DIR / "signal_perf_put_loss_bucket_v0_1.csv", index=False)
put_tail_loss_df.to_csv(AUDIT_DIR / "signal_perf_worst_50_put_losses_v0_1.csv", index=False)

put_expectancy_by_tenor_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tenor_v0_1.csv", index=False)
put_expectancy_by_tier_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tier_v0_1.csv", index=False)
put_expectancy_by_tenor_group_tier_df.to_csv(AUDIT_DIR / "signal_perf_put_expectancy_by_tenor_group_tier_v0_1.csv", index=False)

print("Saved signal performance diagnostics to:", AUDIT_DIR)

Saved signal performance diagnostics to: C:\Users\patri\vrp_project\data\audit
